# Single-perceptron decision boundaries

Simulations supporting Figures 3–4 and supplementary analyses of how ribosome availability and binding affinities affect classification.

## Imports and notebook setup


In [1]:
# Colab setup
import os, sys, subprocess
if "google.colab" in sys.modules:
  cmd = "pip install --upgrade biocircuits bokeh-catplot watermark blackcellmagic intersect"
  process = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  stdout, stderr = process.communicate()
# ------

import scipy.integrate
import biocircuits
import numba

import numpy as np
import matplotlib.pyplot as plt

import bokeh.io
import bokeh.plotting
import bokeh.palettes
from bokeh.models import LinearColorMapper, ColorBar
from bokeh.models import Range1d
from bokeh.io import export_svgs

bokeh.io.output_notebook()

Loading BokehJS ...

## Plotting helpers


In [2]:
#@title Auxiliary functions

# Function to do the 2D projection of decision boundaries with normalization
def contourf(p, x, y, z, title=None, palette="Spectral11", normal = True):
    """Make a filled contour plot given x, y, z data given in 2D arrays."""

    # Normalize the z values
    if normal:
      z_min = z.min()
      z_max = z.max()
      z_normalized = (z - z_min) / (z_max - z_min)  # Normalize to range [0, 1]
    else:
      z_normalized = z

    # Add zero padding at the boundaries for better visualization
    N = z_normalized.shape[1]
    z0 = np.c_[z_normalized, np.zeros(N)]
    z0[-1, -1] = 1.  # Ensure the padding contains a value within range

    # Plot the normalized values
    p.image(
        image=[z0],
        x=x.min(),
        y=y.min(),
        dw=(x.max() - x.min()) * (1 + 1 / N),
        dh=x.max() - x.min(),
        palette=palette,
        alpha=0.8,
        level="underlay",
    )

    # Color mapping based on the normalized z0 values
    color = LinearColorMapper(palette=palette, low=z0.min(), high=z0.max(),)
    cb = ColorBar(color_mapper=color, location=(0, 0), width=10,
                  major_label_text_font_size="12pt")
    p.add_layout(cb, 'right')

    return ()

def numerical_decision_boundary(p, x1, x2, output, level=0.05,color='black', dash = 'dashed'):

    # Extract zero-level contour from output for decision boundary
    X1, X2 = np.meshgrid(x1, x2)
    contours = plt.contour(X1, X2, output, levels=[level])
    plt.close()  # prevent displaying the plot

    for segs in contours.allsegs:  # list of contour levels (only 1 level here)
        for seg in segs:           # each seg = one continuous path
            p.line(seg[:, 0], seg[:, 1], line_width=2, color=color, line_dash=dash)

## Perceptron parameters and dynamical model


In [3]:
class TL_Perceptron:
    def __init__(self):
        # Transcription rates (μM/hr)
        self.ap1 = 10.0  # x1-driven
        self.ap2 = 10.0  # x2-driven
        self.ap3 = 10.0  # x3-driven (endogenous demand)

        # Degradation rates
        self.p = 10.0    # hr⁻¹, mRNA decay
        self.d = 1.0     # hr⁻¹, dilution by growth

        # Ribosome–mRNA association (μM⁻¹·hr⁻¹)
        self.a1 = 100.0
        self.a2 = 100.0
        self.a3 = 100.0

        # Ribosome–mRNA dissociation (hr⁻¹)
        self.d1 = 1.0
        self.d2 = 1.0
        self.d3 = 1.0

        # Translation rates (hr⁻¹)
        self.v1 = 1.0
        self.v2 = 1.0
        self.v3 = 1.0

        # Sequestration
        self.b = 200.0   # μM⁻¹·hr⁻¹
        self.u = 1.0     # hr⁻¹

        # Ribosome pool
        self.rtot = 1.8  # μM

        # Input signals
        self.x1 = 1.0
        self.x2 = 1.0
        self.x3 = 0.0   # endogenous demand, off by default

In [4]:
def Seq_TL_ODEs(X, t, pm):
    m1, m2, m3, r1, r2, r3, y1, y2, y3, c = X

    r = pm.rtot - (r1 + r2 + r3)

    return [
        # Transcription - Translation (TXTL)
        pm.ap1*pm.x1 - pm.p*m1 - pm.d*m1 - pm.a1*m1*r + pm.d1*r1 + pm.v1*r1,
        pm.ap2*pm.x2 - pm.p*m2 - pm.d*m2 - pm.a2*m2*r + pm.d2*r2 + pm.v2*r2,
        pm.ap3*pm.x3 - pm.p*m3 - pm.d*m3 - pm.a3*m3*r + pm.d3*r3 + pm.v3*r3,
        pm.a1*m1*r - pm.d1*r1 - pm.v1*r1 - pm.d*r1 - pm.p*r1,
        pm.a2*m2*r - pm.d2*r2 - pm.v2*r2 - pm.d*r2 - pm.p*r2,
        pm.a3*m3*r - pm.d3*r3 - pm.v3*r3 - pm.d*r3 - pm.p*r3,

        # Molecular sequestration
        pm.v1*r1 - pm.d*y1 - pm.b*y1*y2 + pm.u*c,
        pm.v2*r2 - pm.d*y2 - pm.b*y1*y2 + pm.u*c,
        pm.v3*r3 - pm.d*y3,
        pm.b*y1*y2 - pm.u*c - pm.d*c,
    ]

## Homogeneous binding affinities: decision boundaries across endogenous demand levels


In [5]:
# Color palette
palette = bokeh.palettes.grey(15)[::-1]

# Simulation time (hours)
t = np.linspace(0, 10, 100)

pm = TL_Perceptron()

N = 25
x1 = np.linspace(0, 1, N)  # [uM]
x2 = np.linspace(0, 1, N)  # [uM]

# Set up figures
fig_size = [320, 225]
plots = []
for i in range(3):
    plots.append(bokeh.plotting.figure(width=fig_size[0], height=fig_size[1],
                            x_range=bokeh.models.Range1d(x1[0], x1[-1]),
                            y_range=bokeh.models.Range1d(x2[0], x2[-1]),
                           ))
    plots[i].axis.major_label_text_font_size = '13pt'

# No competition for ribosomes
pm.x3 = 0.0
output1 = np.zeros((N, N))
for i, x1i in enumerate(x1):
    for j, x2j in enumerate(x2):
        pm.x1 = x1i
        pm.x2 = x2j
        y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(10), t, args=(pm,))
        output1[j, i] = y_sol[-1, 6]

# Limited competition (x3 = 3)
pm.x3 = 3.0
output2 = np.zeros((N, N))
for i, x1i in enumerate(x1):
    for j, x2j in enumerate(x2):
        pm.x1 = x1i
        pm.x2 = x2j
        y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(10), t, args=(pm,))
        output2[j, i] = y_sol[-1, 6]

# High endogenous demand (x3 = 5)
pm.x3 = 5.0
output3 = np.zeros((N, N))
for i, x1i in enumerate(x1):
    for j, x2j in enumerate(x2):
        pm.x1 = x1i
        pm.x2 = x2j
        y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(10), t, args=(pm,))
        output3[j, i] = y_sol[-1, 6]

# Normalize by global maximum across all simulations
max_rc = max(output1.max(), output2.max(), output3.max())

contourf(plots[0], x1, x2, output1/max_rc, palette=palette, normal=False)
numerical_decision_boundary(plots[0], x1, x2, output1, level=0.05)
contourf(plots[1], x1, x2, output2/max_rc, palette=palette, normal=False)
numerical_decision_boundary(plots[1], x1, x2, output2, level=0.05)
contourf(plots[2], x1, x2, output3/max_rc, palette=palette, normal=False)
numerical_decision_boundary(plots[2], x1, x2, output3, level=0.05)

for i in range(3):
    plots[i].output_backend = 'svg'

bokeh.io.show(bokeh.layouts.row(plots))

## Free ribosome fraction and decision boundaries at a selected ribosome supply


In [6]:
# Color palette for r_n heatmap
palette_rstar = bokeh.palettes.Blues9[::-1]

# Simulation
t = np.linspace(0, 10, 100)
K_base = 1.1375e-05
v = 3
N = 25
x1_grid = np.linspace(0, 1, N)
x2_grid = np.linspace(0, 1, N)

# x3 value for this row
x3_val = 1.0

# --- Left panel: r_n heatmap with onset line (both heterogeneous) ---
fig_A1 = bokeh.plotting.figure(
    width=320, height=225,
    x_range=bokeh.models.Range1d(0, 1),
    y_range=bokeh.models.Range1d(0, 1),
    x_axis_label='x_1', y_axis_label='x_2'
)
fig_A1.axis.major_label_text_font_size = '13pt'

# Heterogeneous configuration (e_1 < e_2, M_1 binds more strongly)
pm = TL_Perceptron()
pm.x3 = x3_val
pm.rtot = 2.2
pm.a1 = (pm.d1 + pm.v1 + pm.d + pm.p) / (K_base / v)
pm.a2 = (pm.d2 + pm.v2 + pm.d + pm.p) / K_base
pm.a3 = (pm.d3 + pm.v3 + pm.d + pm.p) / K_base

# Single sweep: compute r_n, R*, and demand at each grid point
# (all under the heterogeneous configuration)
output_rn = np.zeros((N, N))
output_Rstar = np.zeros((N, N))
output_demand = np.zeros((N, N))

for i, x1i in enumerate(x1_grid):
    for j, x2j in enumerate(x2_grid):
        pm.x1 = x1i; pm.x2 = x2j
        y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(10), t, args=(pm,))
        r_free = pm.rtot - (y_sol[-1, 3] + y_sol[-1, 4] + y_sol[-1, 5])
        r_n = r_free / pm.rtot
        e3 = K_base / pm.rtot
        S3 = r_n / (r_n + e3)
        Omega1 = (pm.ap1 * x1i) / ((pm.d + pm.p) * pm.rtot)
        Omega2 = (pm.ap2 * x2j) / ((pm.d + pm.p) * pm.rtot)
        Omega3 = (pm.ap3 * pm.x3) / ((pm.d + pm.p) * pm.rtot)
        output_rn[j, i] = r_n
        output_Rstar[j, i] = 1.0 - S3 * Omega3
        output_demand[j, i] = Omega1 + Omega2

# Heatmap of r_n
contourf(fig_A1, x1_grid, x2_grid, output_rn, palette=palette_rstar, normal=True)

# Onset line: zero contour of (R* - Omega), in the heterogeneous configuration
output_onset = output_Rstar - output_demand
numerical_decision_boundary(fig_A1, x1_grid, x2_grid, output_onset,
                            level=0.0, color='black', dash='dashed')

# --- Right panel: three decision boundaries ---
fig_A2 = bokeh.plotting.figure(
    width=250, height=225,
    x_range=bokeh.models.Range1d(0, 1),
    y_range=bokeh.models.Range1d(0, 1),
    x_axis_label='x_1', y_axis_label='x_2'
)
fig_A2.axis.major_label_text_font_size = '13pt'

# Overlay the onset line first (so it sits below the boundaries visually)
numerical_decision_boundary(fig_A2, x1_grid, x2_grid, output_onset,
                            level=0.0, color='black', dash='dashed')

# Homogeneous case (grey)
pm.x3 = x3_val
pm.a1 = (pm.d1 + pm.v1 + pm.d + pm.p) / K_base
pm.a2 = (pm.d2 + pm.v2 + pm.d + pm.p) / K_base
pm.a3 = (pm.d3 + pm.v3 + pm.d + pm.p) / K_base

output_hom = np.zeros((N, N))
for i, x1i in enumerate(x1_grid):
    for j, x2j in enumerate(x2_grid):
        pm.x1 = x1i; pm.x2 = x2j
        y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(10), t, args=(pm,))
        output_hom[j, i] = y_sol[-1, 6]
numerical_decision_boundary(fig_A2, x1_grid, x2_grid, output_hom,
                            level=0.05, color='grey')

# e_1 < e_2 (strong M_1, upward bending) - blue
pm.a1 = (pm.d1 + pm.v1 + pm.d + pm.p) / (K_base / v)
pm.a2 = (pm.d2 + pm.v2 + pm.d + pm.p) / K_base
output_up = np.zeros((N, N))
for i, x1i in enumerate(x1_grid):
    for j, x2j in enumerate(x2_grid):
        pm.x1 = x1i; pm.x2 = x2j
        y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(10), t, args=(pm,))
        output_up[j, i] = y_sol[-1, 6]
numerical_decision_boundary(fig_A2, x1_grid, x2_grid, output_up,
                            level=0.05, color='blue')

# e_1 > e_2 (strong M_2, downward bending) - orange
pm.a1 = (pm.d1 + pm.v1 + pm.d + pm.p) / K_base
pm.a2 = (pm.d2 + pm.v2 + pm.d + pm.p) / (K_base / v)
output_down = np.zeros((N, N))
for i, x1i in enumerate(x1_grid):
    for j, x2j in enumerate(x2_grid):
        pm.x1 = x1i; pm.x2 = x2j
        y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(10), t, args=(pm,))
        output_down[j, i] = y_sol[-1, 6]
numerical_decision_boundary(fig_A2, x1_grid, x2_grid, output_down,
                            level=0.05, color='orange')

for f in [fig_A1, fig_A2]:
    f.output_backend = 'svg'
bokeh.io.show(bokeh.layouts.row(fig_A1, fig_A2))

## Decision-boundary sensitivity to the binding-affinity scale


In [7]:
import numpy as np
import scipy.integrate
import bokeh
from bokeh.plotting import figure, show
from bokeh.layouts import row
import bokeh.palettes

# Palettes: dark → light (matching the convention from earlier panels)
palette_orange = bokeh.palettes.Blues9[::-1][:6]      # 6 shades, dark to light
palette_blue = bokeh.palettes.Oranges9[::-1][:6]

t = np.linspace(0, 10, 100)

# Heterogeneity ratio (same as Fig. 4)
v_factor = 3

# Sweep e_base from strong (Fig. 4 value) to weak (more realistic)
e_base_values = np.logspace(np.log10(6.5e-6), np.log10(6.5e-3), 6)

# Grid
N = 25
x1_grid = np.linspace(0, 1, N)
x2_grid = np.linspace(0, 1, N)

fig_size = [250, 225]
plots = []
for _ in range(2):
    p = bokeh.plotting.figure(
        width=fig_size[0],
        height=fig_size[1],
        x_range=bokeh.models.Range1d(x1_grid[0], x1_grid[-1]),
        y_range=bokeh.models.Range1d(x2_grid[0], x2_grid[-1]),
        x_axis_label='x_1', y_axis_label='x_2'
    )
    p.axis.major_label_text_font_size = '13pt'
    plots.append(p)


def set_e_values(pm, e1, e2, e3):
    """Convert dimensionless e_i to a_i using the manuscript convention."""
    pm.a1 = (pm.d1 + pm.v1 + pm.d + pm.p) / (e1 * pm.rtot)
    pm.a2 = (pm.d2 + pm.v2 + pm.d + pm.p) / (e2 * pm.rtot)
    pm.a3 = (pm.d3 + pm.v3 + pm.d + pm.p) / (e3 * pm.rtot)

# =========================
# Left panel: e_1 > e_2 = e_3 (downward bending in (x_1, x_2) plane)
# =========================
print("Left panel (e_1 > e_2 = e_3):")
print(f"{'e_base':>12} {'e_1':>12} {'e_2':>12} {'e_3':>12}")
for e_base, color in zip(e_base_values, palette_blue):
    pm = TL_Perceptron()
    pm.x3 = 1.0
    e1 = v_factor * e_base
    e2 = e_base
    e3 = e_base
    set_e_values(pm, e1, e2, e3)
    print(f"{e_base:>12.4e} {e1:>12.4e} {e2:>12.4e} {e3:>12.4e}")

    output = np.zeros((N, N))
    for i, x1i in enumerate(x1_grid):
        for j, x2j in enumerate(x2_grid):
            pm.x1 = x1i
            pm.x2 = x2j
            y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(10), t, args=(pm,))
            output[j, i] = y_sol[-1, 6]
    numerical_decision_boundary(
        plots[0],
        x1_grid, x2_grid,
        output,
        level=0.05,
        color=color
    )

# =========================
# Right panel: e_2 > e_1 = e_3 (upward bending in (x_1, x_2) plane)
# =========================
print("\nRight panel (e_2 > e_1 = e_3):")
print(f"{'e_base':>12} {'e_1':>12} {'e_2':>12} {'e_3':>12}")
for e_base, color in zip(e_base_values, palette_orange):
    pm = TL_Perceptron()
    pm.x3 = 1.0
    e1 = e_base
    e2 = v_factor * e_base
    e3 = e_base
    set_e_values(pm, e1, e2, e3)
    print(f"{e_base:>12.4e} {e1:>12.4e} {e2:>12.4e} {e3:>12.4e}")

    output = np.zeros((N, N))
    for i, x1i in enumerate(x1_grid):
        for j, x2j in enumerate(x2_grid):
            pm.x1 = x1i
            pm.x2 = x2j
            y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(10), t, args=(pm,))
            output[j, i] = y_sol[-1, 6]
    numerical_decision_boundary(
        plots[1],
        x1_grid, x2_grid,
        output,
        level=0.05,
        color=color
    )

for p in plots:
    p.output_backend = "svg"

show(row(plots[0], plots[1]))

Left panel (e_1 > e_2 = e_3):
      e_base          e_1          e_2          e_3
  6.5000e-06   1.9500e-05   6.5000e-06   6.5000e-06
  2.5877e-05   7.7631e-05   2.5877e-05   2.5877e-05
  1.0302e-04   3.0905e-04   1.0302e-04   1.0302e-04
  4.1012e-04   1.2304e-03   4.1012e-04   4.1012e-04
  1.6327e-03   4.8982e-03   1.6327e-03   1.6327e-03
  6.5000e-03   1.9500e-02   6.5000e-03   6.5000e-03

Right panel (e_2 > e_1 = e_3):
      e_base          e_1          e_2          e_3
  6.5000e-06   6.5000e-06   1.9500e-05   6.5000e-06
  2.5877e-05   2.5877e-05   7.7631e-05   2.5877e-05
  1.0302e-04   1.0302e-04   3.0905e-04   1.0302e-04
  4.1012e-04   4.1012e-04   1.2304e-03   4.1012e-04
  1.6327e-03   1.6327e-03   4.8982e-03   1.6327e-03
  6.5000e-03   6.5000e-03   1.9500e-02   6.5000e-03


## Heterogeneous binding affinities: output maps and decision boundaries


In [8]:
# Color palette
palette = bokeh.palettes.grey(15)[::-1]

# Simulation time (hours)
t = np.linspace(0, 10, 100)

pm = TL_Perceptron()
pm.x3 = 1.0

# ==========================================
# Directly specify e-values
# ==========================================

e_base = 6.5e-6
v_factor = 3

# Total ribosome pool used in the definition of e
RT = pm.rtot

N = 25
x1 = np.linspace(0, 1, N)
x2 = np.linspace(0, 1, N)

fig_size = [320, 225]
plots = []

for _ in range(2):
    p = bokeh.plotting.figure(
        width=fig_size[0],
        height=fig_size[1],
        x_range=bokeh.models.Range1d(x1[0], x1[-1]),
        y_range=bokeh.models.Range1d(x2[0], x2[-1]),
    )
    p.axis.major_label_text_font_size = '13pt'
    plots.append(p)


def set_e_values(pm, e1, e2, e3):
    """
    Convert desired e-values into association rates a_i.
    Assumes x_ref = 1 in the definition of e.
    """

    pm.a1 = (pm.d1 + pm.v1 + pm.d + pm.p) / (e1 * RT)
    pm.a2 = (pm.d2 + pm.v2 + pm.d + pm.p) / (e2 * RT)
    pm.a3 = (pm.d3 + pm.v3 + pm.d + pm.p) / (e3 * RT)


# =====================================================
# Case 1: e1 > e2 = e3
# =====================================================

set_e_values(
    pm,
    e1=v_factor * e_base,
    e2=e_base,
    e3=e_base,
)

output1 = np.zeros((N, N))

for i, x1i in enumerate(x1):
    for j, x2j in enumerate(x2):

        pm.x1 = x1i
        pm.x2 = x2j

        y_sol = scipy.integrate.odeint(
            Seq_TL_ODEs,
            np.zeros(10),
            t,
            args=(pm,)
        )

        output1[j, i] = y_sol[-1, 6]


# =====================================================
# Case 2: e2 > e1 = e3
# =====================================================

set_e_values(
    pm,
    e1=e_base,
    e2=v_factor * e_base,
    e3 = 6.5e-1#e_base
)

output2 = np.zeros((N, N))

for i, x1i in enumerate(x1):
    for j, x2j in enumerate(x2):

        pm.x1 = x1i
        pm.x2 = x2j

        y_sol = scipy.integrate.odeint(
            Seq_TL_ODEs,
            np.zeros(10),
            t,
            args=(pm,)
        )

        output2[j, i] = y_sol[-1, 6]


# =====================================================
# Plotting
# =====================================================

max_rc = max(output1.max(), output2.max())

contourf(
    plots[0],
    x1,
    x2,
    output1 / max_rc,
    palette=palette,
    normal=False
)

numerical_decision_boundary(
    plots[0],
    x1,
    x2,
    output1,
    level=0.05
)

contourf(
    plots[1],
    x1,
    x2,
    output2 / max_rc,
    palette=palette,
    normal=False
)

numerical_decision_boundary(
    plots[1],
    x1,
    x2,
    output2,
    level=0.05
)

for p in plots:
    p.output_backend = "svg"

bokeh.io.show(
    bokeh.layouts.row(plots)
)

## Resource-competition onset and affinity-dependent boundary bending


In [9]:
# Simulation time (hours)
t = np.linspace(0, 10, 100)

pm = TL_Perceptron()
pm.x3 = 1.0

K_base = 1.1375e-05
v = 3

N = 25
x1 = np.linspace(0, 1, N)
x2 = np.linspace(0, 1, N)

fig = bokeh.plotting.figure(
    width=250, height=225,
    x_range=bokeh.models.Range1d(x1[0], x1[-1]),
    y_range=bokeh.models.Range1d(x2[0], x2[-1]),
    x_axis_label='x1',
    y_axis_label='x2'
)
fig.axis.major_label_text_font_size = '13pt'

# ============================================================================
# COMPUTE ONSET USING R* (effective availability)
# ============================================================================
pm.a1 = (pm.d1 + pm.v1 + pm.d + pm.p) / K_base
pm.a2 = (pm.d2 + pm.v2 + pm.d + pm.p) / K_base
pm.a3 = (pm.d3 + pm.v3 + pm.d + pm.p) / K_base

output_Rstar = np.zeros((N, N))
for i, x1i in enumerate(x1):
    for j, x2j in enumerate(x2):
        pm.x1 = x1i
        pm.x2 = x2j
        y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(10), t, args=(pm,))
        r1_final = y_sol[-1, 3]
        r2_final = y_sol[-1, 4]
        r3_final = y_sol[-1, 5]
        r_free = pm.rtot - (r1_final + r2_final + r3_final)
        r_n = r_free / pm.rtot

        e3 = K_base / pm.rtot
        Omega3 = (pm.ap3 * pm.x3) / ((pm.d + pm.p) * pm.rtot)
        S3 = r_n / (r_n + e3)
        output_Rstar[j, i] = 1.0 - S3 * Omega3

output_demand = np.zeros((N, N))
for i, x1i in enumerate(x1):
    for j, x2j in enumerate(x2):
        e1 = K_base / pm.rtot
        e2 = K_base / pm.rtot
        Omega1 = (pm.ap1 * x1i) / ((pm.d + pm.p) * pm.rtot)
        Omega2 = (pm.ap2 * x2j) / ((pm.d + pm.p) * pm.rtot)
        output_demand[j, i] = Omega1 + Omega2

output_onset = output_Rstar - output_demand
numerical_decision_boundary(fig, x1, x2, output_onset, level=0.0, color='black', dash='dashed')

# ============================================================================
# CASE 1: Homogeneous case
# ============================================================================
pm.a1 = (pm.d1 + pm.v1 + pm.d + pm.p) / K_base
pm.a2 = (pm.d2 + pm.v2 + pm.d + pm.p) / K_base
pm.a3 = (pm.d3 + pm.v3 + pm.d + pm.p) / K_base

output1 = np.zeros((N, N))
for i, x1i in enumerate(x1):
    for j, x2j in enumerate(x2):
        pm.x1 = x1i
        pm.x2 = x2j
        y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(10), t, args=(pm,))
        output1[j, i] = y_sol[-1, 6]

numerical_decision_boundary(fig, x1, x2, output1, level=0.05, color='grey')

# ============================================================================
# CASE 2: Strong binding for mRNA 1 (x1 favored) - GREEN
# ============================================================================
pm.a1 = (pm.d1 + pm.v1 + pm.d + pm.p) / (K_base / v)
pm.a2 = (pm.d2 + pm.v2 + pm.d + pm.p) / K_base
pm.a3 = (pm.d3 + pm.v3 + pm.d + pm.p) / K_base

output2 = np.zeros((N, N))
for i, x1i in enumerate(x1):
    for j, x2j in enumerate(x2):
        pm.x1 = x1i
        pm.x2 = x2j
        y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(10), t, args=(pm,))
        output2[j, i] = y_sol[-1, 6]

numerical_decision_boundary(fig, x1, x2, output2, level=0.05, color='green')

# ============================================================================
# CASE 3: Strong binding for mRNA 2 (x2 favored) - RED
# ============================================================================
pm.a1 = (pm.d1 + pm.v1 + pm.d + pm.p) / K_base
pm.a2 = (pm.d2 + pm.v2 + pm.d + pm.p) / (K_base / v)
pm.a3 = (pm.d3 + pm.v3 + pm.d + pm.p) / K_base

output3 = np.zeros((N, N))
for i, x1i in enumerate(x1):
    for j, x2j in enumerate(x2):
        pm.x1 = x1i
        pm.x2 = x2j
        y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(10), t, args=(pm,))
        output3[j, i] = y_sol[-1, 6]

numerical_decision_boundary(fig, x1, x2, output3, level=0.05, color='red')

fig.output_backend = 'svg'
bokeh.io.show(fig)

## Uncertainty regions under binding-affinity variation


In [10]:
# ============================================================================
# PARAMETERS
# ============================================================================
t = np.linspace(0, 10, 100)

pm = TL_Perceptron()

# Grid resolution
N = 25
x1 = np.linspace(0, 1, N)
x2 = np.linspace(0, 1, N)

# Variation percentages
vr = [0.25, 0.5]  # 30% and 60%

# ============================================================================
# LEFT PLOT
# ============================================================================
K_base = 1.1375e-05
pm.rtot = 1.8
pm.x3 = 1.0

p_left = bokeh.plotting.figure(
    width=250, height=225,
    x_range=(0, 1), y_range=(0, 1),
    x_axis_label="x₁", y_axis_label="x₂",
)

variations = [1 + vr[1], 1 + vr[0], 1 - vr[0], 1 - vr[1]]
boundaries_left = []

for var_factor in variations:
    pm.a1 = (pm.d1 + pm.v1 + pm.d + pm.p) / (K_base * var_factor)
    pm.a2 = (pm.d2 + pm.v2 + pm.d + pm.p) / K_base
    pm.a3 = (pm.d3 + pm.v3 + pm.d + pm.p) / K_base

    output = np.zeros((N, N))
    for i, x1i in enumerate(x1):
        for j, x2j in enumerate(x2):
            pm.x1 = x1i
            pm.x2 = x2j
            y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(10), t, args=(pm,))
            output[j, i] = y_sol[-1, 6]

    contour = plt.contour(x1, x2, output, levels=[0.05])
    segs = contour.allsegs[0]
    if len(segs) > 0:
        vertices = max(segs, key=lambda v: v.shape[0])
        boundaries_left.append(vertices)
    plt.close()

# ---- SHADED REGIONS ----
if len(boundaries_left) >= 4:
    path_hi_60 = boundaries_left[0]
    path_lo_60 = boundaries_left[3]

    if path_lo_60[0, 0] > path_lo_60[-1, 0]:
        path_lo_60 = path_lo_60[::-1]
    if path_hi_60[0, 0] > path_hi_60[-1, 0]:
        path_hi_60 = path_hi_60[::-1]

    xs = np.concatenate([path_lo_60[:, 0], path_hi_60[::-1, 0]])
    ys = np.concatenate([path_lo_60[:, 1], path_hi_60[::-1, 1]])
    p_left.patch(xs, ys, color="orange", alpha=0.6, line_alpha=0, legend_label="±60%")

if len(boundaries_left) >= 3:
    path_hi_30 = boundaries_left[1]
    path_lo_30 = boundaries_left[2]

    if path_lo_30[0, 0] > path_lo_30[-1, 0]:
        path_lo_30 = path_lo_30[::-1]
    if path_hi_30[0, 0] > path_hi_30[-1, 0]:
        path_hi_30 = path_hi_30[::-1]

    xs = np.concatenate([path_lo_30[:, 0], path_hi_30[::-1, 0]])
    ys = np.concatenate([path_lo_30[:, 1], path_hi_30[::-1, 1]])
    p_left.patch(xs, ys, color='#4C72B0', alpha=0.4, line_alpha=0, legend_label="±30%")

for boundary in boundaries_left:
    p_left.line(boundary[:, 0], boundary[:, 1],
                line_color="black", line_dash="dashed", line_width=2)

# ============================================================================
# COMPUTE ONSET USING R* — homogeneous affinities (baseline reference)
# ============================================================================
pm.a1 = (pm.d1 + pm.v1 + pm.d + pm.p) / K_base
pm.a2 = (pm.d2 + pm.v2 + pm.d + pm.p) / K_base
pm.a3 = (pm.d3 + pm.v3 + pm.d + pm.p) / K_base

output_Rstar = np.zeros((N, N))
for i, x1i in enumerate(x1):
    for j, x2j in enumerate(x2):
        pm.x1 = x1i
        pm.x2 = x2j
        y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(10), t, args=(pm,))
        r1_final = y_sol[-1, 3]
        r2_final = y_sol[-1, 4]
        r3_final = y_sol[-1, 5]
        r_free = pm.rtot - (r1_final + r2_final + r3_final)
        r_n = r_free / pm.rtot

        e3 = K_base / pm.rtot
        S3 = r_n / (r_n + e3)
        Omega3 = (pm.ap3 * pm.x3) / ((pm.d + pm.p) * pm.rtot)
        output_Rstar[j, i] = 1.0 - S3 * Omega3

output_demand = np.zeros((N, N))
for i, x1i in enumerate(x1):
    for j, x2j in enumerate(x2):
        Omega1 = (pm.ap1 * x1i) / ((pm.d + pm.p) * pm.rtot)
        Omega2 = (pm.ap2 * x2j) / ((pm.d + pm.p) * pm.rtot)
        output_demand[j, i] = Omega1 + Omega2

output_onset = output_Rstar - output_demand
numerical_decision_boundary(p_left, x1, x2, output_onset,
                            level=0.0, color='black', dash='dashed')

# ============================================================================
# RIGHT PLOT
# ============================================================================
pm.rtot = 1.8
pm.x3 = 5.0

p_right = bokeh.plotting.figure(
    width=250, height=225,
    x_range=(0, 1), y_range=(0, 1),
    x_axis_label="x₁", y_axis_label="x₂",
)

vr_right = [0.1, 0.3]
variations_right = [1 + vr_right[1], 1 + vr_right[0],
                    1 - vr_right[0], 1 - vr_right[1]]
boundaries_right = []

for var_factor in variations_right:
    pm.a1 = (pm.d1 + pm.v1 + pm.d + pm.p) / (K_base * var_factor)
    pm.a2 = (pm.d2 + pm.v2 + pm.d + pm.p) / K_base
    pm.a3 = (pm.d3 + pm.v3 + pm.d + pm.p) / K_base

    output = np.zeros((N, N))
    for i, x1i in enumerate(x1):
        for j, x2j in enumerate(x2):
            pm.x1 = x1i
            pm.x2 = x2j
            y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(10), t, args=(pm,))
            output[j, i] = y_sol[-1, 6]

    contour = plt.contour(x1, x2, output, levels=[0.05])
    segs = contour.allsegs[0]
    if len(segs) > 0:
        vertices = max(segs, key=lambda v: v.shape[0])
        boundaries_right.append(vertices)
    plt.close()

# ---- SHADED REGIONS ----
if len(boundaries_right) >= 4:
    hi = boundaries_right[0]
    lo = boundaries_right[3]

    if lo[0, 0] > lo[-1, 0]:
        lo = lo[::-1]
    if hi[0, 0] > hi[-1, 0]:
        hi = hi[::-1]

    xs = np.concatenate([lo[:, 0], hi[::-1, 0]])
    ys = np.concatenate([lo[:, 1], hi[::-1, 1]])
    p_right.patch(xs, ys, color="orange", alpha=0.6, line_alpha=0, legend_label="±30%")

if len(boundaries_right) >= 3:
    hi = boundaries_right[1]
    lo = boundaries_right[2]

    if lo[0, 0] > lo[-1, 0]:
        lo = lo[::-1]
    if hi[0, 0] > hi[-1, 0]:
        hi = hi[::-1]

    xs = np.concatenate([lo[:, 0], hi[::-1, 0]])
    ys = np.concatenate([lo[:, 1], hi[::-1, 1]])
    p_right.patch(xs, ys, color='#4C72B0', alpha=0.4, line_alpha=0, legend_label="±10%")

for boundary in boundaries_right:
    p_right.line(boundary[:, 0], boundary[:, 1],
                 line_color="black", line_dash="dashed", line_width=2)

# ---- ONSET FOR RIGHT PLOT (x3 = 5) ----
pm.a1 = (pm.d1 + pm.v1 + pm.d + pm.p) / K_base
pm.a2 = (pm.d2 + pm.v2 + pm.d + pm.p) / K_base
pm.a3 = (pm.d3 + pm.v3 + pm.d + pm.p) / K_base

output_Rstar_right = np.zeros((N, N))
for i, x1i in enumerate(x1):
    for j, x2j in enumerate(x2):
        pm.x1 = x1i
        pm.x2 = x2j
        y_sol = scipy.integrate.odeint(Seq_TL_ODEs, np.zeros(10), t, args=(pm,))
        r1_final = y_sol[-1, 3]
        r2_final = y_sol[-1, 4]
        r3_final = y_sol[-1, 5]
        r_free = pm.rtot - (r1_final + r2_final + r3_final)
        r_n = r_free / pm.rtot

        e3 = K_base / pm.rtot
        S3 = r_n / (r_n + e3)
        Omega3 = (pm.ap3 * pm.x3) / ((pm.d + pm.p) * pm.rtot)
        output_Rstar_right[j, i] = 1.0 - S3 * Omega3

output_onset_right = output_Rstar_right - output_demand
numerical_decision_boundary(p_right, x1, x2, output_onset_right,
                            level=0.0, color='black', dash='dashed')

# ============================================================================
# FINAL DISPLAY
# ============================================================================
p_left.axis.major_label_text_font_size = '13pt'
p_right.axis.major_label_text_font_size = '13pt'
p_left.output_backend = "svg"
p_right.output_backend = "svg"

bokeh.io.show(bokeh.layouts.row(p_left, p_right))